# **Model Training - UK Property Price Prediction**

This notebook implements multiple ML algorithms using PySpark MLlib.

**Objectives:**
- Train multiple regression models
- Implement hyperparameter tuning
- Compare distributed vs single-node performance
- Analyze scalability
- Save trained models

In [1]:
import os
import time
import pickle
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, rand
from pyspark.ml.regression import (
    LinearRegression, DecisionTreeRegressor, 
    RandomForestRegressor, GBTRegressor
)
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml import Pipeline, PipelineModel
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

## **1. Initialize Spark Session**

In [2]:
spark = SparkSession.builder \
    .appName("Model_Training") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.executor.cores", "4") \
    .config("spark.executor.instances", "2") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

log4j = spark.sparkContext._jvm.org.apache.log4j
log4j.LogManager.getLogger("org.apache.spark.storage.MemoryStore").setLevel(log4j.Level.ERROR)
log4j.LogManager.getLogger("org.apache.spark.storage.BlockManager").setLevel(log4j.Level.ERROR)
log4j.LogManager.getLogger("org.apache.spark.scheduler.DAGScheduler").setLevel(log4j.Level.ERROR)

print(f"Spark Version: {spark.version}")
print(f"Executor Memory: {spark.conf.get('spark.executor.memory')}")
print(f"Executor Cores: {spark.conf.get('spark.executor.cores')}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/27 12:54:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/27 12:54:07 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/02/27 12:54:07 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Spark Version: 3.5.0
Executor Memory: 8g
Executor Cores: 4


## **2. Load Processed Features**

In [3]:
df = spark.read.parquet("../data/processed_features.parquet")

print(f"Data loaded: {df.count():,} records")
print(f"Partitions: {df.rdd.getNumPartitions()}")

df = df.select("features", "label", "year", "property_type", "postcode_area")
df.cache()
print("Data cached")

Data loaded: 6,713,112 records
Partitions: 8
Data cached


## **3. Train/Validation/Test Split**

In [4]:
df_2019_2022 = df.filter(col("year") <= 2022)
df_2023 = df.filter(col("year") == 2023)
df_2024_2025 = df.filter(col("year") >= 2024)

train_data, val_data = df_2019_2022.randomSplit([0.8, 0.2], seed=42)
test_data = df_2023
future_test = df_2024_2025

print(f"Train: {train_data.count():,} records")
print(f"Validation: {val_data.count():,} records")
print(f"Test (2023): {test_data.count():,} records")
print(f"Future Test (2024-2025): {future_test.count():,} records")

train_data.cache()
val_data.cache()
test_data.cache()
future_test.cache()

Train: 3,407,564 records


Validation: 850,829 records
Test (2023): 856,811 records
Future Test (2024-2025): 1,597,908 records


DataFrame[features: vector, label: int, year: int, property_type: string, postcode_area: string]

## **4. Model 1: Linear Regression**

In [5]:
lr = LinearRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    regParam=0.1,
    elasticNetParam=0.5
)

print("Linear Regression configured")

Linear Regression configured


In [6]:
start_time = time.time()

lr_model = lr.fit(train_data)

lr_train_time = time.time() - start_time
print(f"Linear Regression trained in {lr_train_time:.2f} seconds")

26/02/27 12:55:19 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/02/27 12:55:19 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


Linear Regression trained in 29.03 seconds


In [7]:
lr_predictions = lr_model.transform(test_data)

evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction")

lr_rmse = evaluator.evaluate(lr_predictions, {evaluator.metricName: "rmse"})
lr_r2 = evaluator.evaluate(lr_predictions, {evaluator.metricName: "r2"})
lr_mae = evaluator.evaluate(lr_predictions, {evaluator.metricName: "mae"})

print(f"\nLinear Regression Results:")
print(f"  RMSE: {lr_rmse:,.2f}")
print(f"  R2: {lr_r2:.4f}")
print(f"  MAE: {lr_mae:,.2f}")


Linear Regression Results:
  RMSE: 126,072.82
  R2: 0.9185
  MAE: 50,003.03


In [8]:
lr_model.write().overwrite().save("../data/models/linear_regression")
print("Linear Regression model saved")

Linear Regression model saved


## **5. Model 2: Decision Tree Regressor**

In [9]:
dt = DecisionTreeRegressor(
    featuresCol="features",
    labelCol="label",
    maxDepth=10,
    maxBins=32,
    minInstancesPerNode=100
)

print("Decision Tree configured")

Decision Tree configured


In [10]:
start_time = time.time()

dt_model = dt.fit(train_data)

dt_train_time = time.time() - start_time
print(f"Decision Tree trained in {dt_train_time:.2f} seconds")

26/02/27 12:55:44 WARN MemoryStore: Not enough space to cache rdd_158_5 in memory! (computed 556.2 MiB so far)


Decision Tree trained in 34.15 seconds


In [11]:
dt_predictions = dt_model.transform(test_data)

dt_rmse = evaluator.evaluate(dt_predictions, {evaluator.metricName: "rmse"})
dt_r2 = evaluator.evaluate(dt_predictions, {evaluator.metricName: "r2"})
dt_mae = evaluator.evaluate(dt_predictions, {evaluator.metricName: "mae"})

print(f"\nDecision Tree Results:")
print(f"  RMSE: {dt_rmse:,.2f}")
print(f"  R2: {dt_r2:.4f}")
print(f"  MAE: {dt_mae:,.2f}")


Decision Tree Results:
  RMSE: 121,368.46
  R2: 0.9244
  MAE: 23,977.05


In [12]:
dt_feature_importance = dt_model.featureImportances
print(f"Feature importance computed: {len(dt_feature_importance)} features")

Feature importance computed: 306 features


In [13]:
dt_model.write().overwrite().save("../data/models/decision_tree")
print("Decision Tree model saved")

Decision Tree model saved


## **6. Model 3: Random Forest Regressor**

In [14]:
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="label",
    numTrees=50,
    maxDepth=10,
    maxBins=32,
    minInstancesPerNode=100,
    subsamplingRate=0.8,
    seed=42
)

print("Random Forest configured with 50 trees")

Random Forest configured with 50 trees


In [15]:
start_time = time.time()

rf_model = rf.fit(train_data)

rf_train_time = time.time() - start_time
print(f"Random Forest trained in {rf_train_time:.2f} seconds")

26/02/27 12:56:43 WARN MemoryStore: Not enough space to cache rdd_251_7 in memory! (computed 358.2 MiB so far)
26/02/27 12:56:45 WARN MemoryStore: Not enough space to cache rdd_251_7 in memory! (computed 358.2 MiB so far)
26/02/27 12:57:14 WARN MemoryStore: Not enough space to cache rdd_251_7 in memory! (computed 158.1 MiB so far)
26/02/27 12:57:47 WARN MemoryStore: Not enough space to cache rdd_251_7 in memory! (computed 158.1 MiB so far)
26/02/27 12:58:24 WARN MemoryStore: Not enough space to cache rdd_251_7 in memory! (computed 158.1 MiB so far)
26/02/27 12:59:03 WARN MemoryStore: Not enough space to cache rdd_251_7 in memory! (computed 158.1 MiB so far)
26/02/27 12:59:47 WARN MemoryStore: Not enough space to cache rdd_251_7 in memory! (computed 158.1 MiB so far)
26/02/27 13:00:31 WARN MemoryStore: Not enough space to cache rdd_251_7 in memory! (computed 158.1 MiB so far)
26/02/27 13:01:20 WARN MemoryStore: Not enough space to cache rdd_251_7 in memory! (computed 158.1 MiB so far)
2

Random Forest trained in 447.05 seconds


In [16]:
rf_predictions = rf_model.transform(test_data)

rf_rmse = evaluator.evaluate(rf_predictions, {evaluator.metricName: "rmse"})
rf_r2 = evaluator.evaluate(rf_predictions, {evaluator.metricName: "r2"})
rf_mae = evaluator.evaluate(rf_predictions, {evaluator.metricName: "mae"})

print(f"\nRandom Forest Results:")
print(f"  RMSE: {rf_rmse:,.2f}")
print(f"  R2: {rf_r2:.4f}")
print(f"  MAE: {rf_mae:,.2f}")


Random Forest Results:
  RMSE: 102,612.38
  R2: 0.9460
  MAE: 20,003.07


In [17]:
rf_feature_importance = rf_model.featureImportances
print(f"Feature importance computed")

Feature importance computed


In [18]:
rf_model.write().overwrite().save("../data/models/random_forest")
print("Random Forest model saved")

Random Forest model saved


## **7. Model 4: Gradient Boosted Trees**

In [19]:
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="label",
    maxIter=30,
    maxDepth=8,
    maxBins=32,
    stepSize=0.1,
    subsamplingRate=0.8,
    seed=42
)

print("Gradient Boosted Trees configured with 30 iterations")

Gradient Boosted Trees configured with 30 iterations


In [20]:
start_time = time.time()

gbt_model = gbt.fit(train_data)

gbt_train_time = time.time() - start_time
print(f"GBT trained in {gbt_train_time:.2f} seconds")

26/02/27 13:03:55 WARN MemoryStore: Not enough space to cache rdd_356_5 in memory! (computed 351.7 MiB so far)


GBT trained in 425.71 seconds


In [21]:
gbt_predictions = gbt_model.transform(test_data)

gbt_rmse = evaluator.evaluate(gbt_predictions, {evaluator.metricName: "rmse"})
gbt_r2 = evaluator.evaluate(gbt_predictions, {evaluator.metricName: "r2"})
gbt_mae = evaluator.evaluate(gbt_predictions, {evaluator.metricName: "mae"})

print(f"\nGradient Boosted Trees Results:")
print(f"  RMSE: {gbt_rmse:,.2f}")
print(f"  R2: {gbt_r2:.4f}")
print(f"  MAE: {gbt_mae:,.2f}")


Gradient Boosted Trees Results:
  RMSE: 98,981.02
  R2: 0.9497
  MAE: 23,584.74


In [22]:
gbt_feature_importance = gbt_model.featureImportances
print(f"Feature importance computed")

Feature importance computed


In [23]:
gbt_model.write().overwrite().save("../data/models/gradient_boosted_trees")
print("GBT model saved")

GBT model saved


## **8. Hyperparameter Tuning - Random Forest**

In [24]:
rf_tuning = RandomForestRegressor(
    featuresCol="features",
    labelCol="label",
    seed=42
)

paramGrid = ParamGridBuilder() \
    .addGrid(rf_tuning.numTrees, [30, 50, 70]) \
    .addGrid(rf_tuning.maxDepth, [8, 10, 12]) \
    .addGrid(rf_tuning.minInstancesPerNode, [50, 100]) \
    .build()

print(f"Parameter grid size: {len(paramGrid)}")

Parameter grid size: 18


In [25]:
cv = CrossValidator(
    estimator=rf_tuning,
    estimatorParamMaps=paramGrid,
    evaluator=RegressionEvaluator(labelCol="label", metricName="rmse"),
    numFolds=3,
    parallelism=4,
    seed=42
)

print("CrossValidator configured with 3 folds")

CrossValidator configured with 3 folds


In [26]:
train_sample = train_data.sample(False, 0.1, seed=42)
print(f"Using sample of {train_sample.count():,} records for tuning")

Using sample of 340,936 records for tuning


In [27]:
start_time = time.time()

cv_model = cv.fit(train_sample)

cv_time = time.time() - start_time
print(f"Cross-validation completed in {cv_time:.2f} seconds")

Cross-validation completed in 1934.67 seconds


In [28]:
best_model = cv_model.bestModel

print(f"\nBest Model Parameters:")
print(f"  Num Trees: {best_model.getNumTrees}")
print(f"  Max Depth: {best_model.getMaxDepth()}")
print(f"  Min Instances Per Node: {best_model.getMinInstancesPerNode()}")


Best Model Parameters:
  Num Trees: 50
  Max Depth: 12
  Min Instances Per Node: 50


In [29]:
cv_predictions = best_model.transform(test_data)

cv_rmse = evaluator.evaluate(cv_predictions, {evaluator.metricName: "rmse"})
cv_r2 = evaluator.evaluate(cv_predictions, {evaluator.metricName: "r2"})
cv_mae = evaluator.evaluate(cv_predictions, {evaluator.metricName: "mae"})

print(f"\nTuned Model Results:")
print(f"  RMSE: {cv_rmse:,.2f}")
print(f"  R2: {cv_r2:.4f}")
print(f"  MAE: {cv_mae:,.2f}")


Tuned Model Results:
  RMSE: 107,918.63
  R2: 0.9402
  MAE: 17,439.91


In [30]:
cv_model.write().overwrite().save("../data/models/tuned_random_forest")
print("Tuned model saved")

26/02/27 13:43:30 WARN TaskSetManager: Stage 2066 contains a task of very large size (1773 KiB). The maximum recommended task size is 1000 KiB.


Tuned model saved


## **9. Model Comparison**

In [31]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree', 'Random Forest', 'GBT', 'Tuned RF'],
    'RMSE': [lr_rmse, dt_rmse, rf_rmse, gbt_rmse, cv_rmse],
    'R2': [lr_r2, dt_r2, rf_r2, gbt_r2, cv_r2],
    'MAE': [lr_mae, dt_mae, rf_mae, gbt_mae, cv_mae],
    'Training_Time': [lr_train_time, dt_train_time, rf_train_time, gbt_train_time, cv_time]
})

print("\n=== Model Comparison ===")
print(results.to_string(index=False))


=== Model Comparison ===
            Model          RMSE       R2          MAE  Training_Time
Linear Regression 126072.820907 0.918453 50003.032929      29.032401
    Decision Tree 121368.464826 0.924425 23977.052159      34.148242
    Random Forest 102612.380020 0.945979 20003.071448     447.051619
              GBT  98981.024417 0.949735 23584.738203     425.711873
         Tuned RF 107918.627321 0.940247 17439.913703    1934.671469


In [32]:
results.to_csv("../data/model_comparison.csv", index=False)
print("\nResults saved to CSV")


Results saved to CSV


## **10. Scalability Analysis**

In [33]:
sample_sizes = [0.1, 0.2, 0.5, 1.0]
scalability_results = []

for size in sample_sizes:
    sample = train_data.sample(False, size, seed=42)
    sample_count = sample.count()
    
    start = time.time()
    model = rf.fit(sample)
    elapsed = time.time() - start
    
    scalability_results.append({
        'sample_fraction': size,
        'sample_size': sample_count,
        'training_time': elapsed
    })
    
    print(f"Size: {size*100:.0f}% ({sample_count:,} records) - Time: {elapsed:.2f}s")

scalability_df = pd.DataFrame(scalability_results)
scalability_df.to_csv("../data/scalability_analysis.csv", index=False)
print("\nScalability analysis saved")

Size: 10% (340,936 records) - Time: 48.29s


Size: 20% (682,352 records) - Time: 92.33s


Size: 50% (1,705,749 records) - Time: 217.84s


26/02/27 13:50:07 WARN MemoryStore: Not enough space to cache rdd_4923_7 in memory! (computed 358.2 MiB so far)
26/02/27 13:50:09 WARN MemoryStore: Not enough space to cache rdd_4923_7 in memory! (computed 358.2 MiB so far)
26/02/27 13:50:39 WARN MemoryStore: Not enough space to cache rdd_4923_7 in memory! (computed 158.1 MiB so far)
26/02/27 13:51:12 WARN MemoryStore: Not enough space to cache rdd_4923_7 in memory! (computed 158.1 MiB so far)
26/02/27 13:51:49 WARN MemoryStore: Not enough space to cache rdd_4923_7 in memory! (computed 158.1 MiB so far)
26/02/27 13:52:27 WARN MemoryStore: Not enough space to cache rdd_4923_7 in memory! (computed 158.1 MiB so far)
26/02/27 13:53:10 WARN MemoryStore: Not enough space to cache rdd_4923_7 in memory! (computed 158.1 MiB so far)
26/02/27 13:53:52 WARN MemoryStore: Not enough space to cache rdd_4923_7 in memory! (computed 158.1 MiB so far)
26/02/27 13:54:40 WARN MemoryStore: Not enough space to cache rdd_4923_7 in memory! (computed 158.1 MiB 

Size: 100% (3,407,564 records) - Time: 429.32s

Scalability analysis saved


## **11. Future Test Performance**

In [34]:
future_predictions = best_model.transform(future_test)

future_rmse = evaluator.evaluate(future_predictions, {evaluator.metricName: "rmse"})
future_r2 = evaluator.evaluate(future_predictions, {evaluator.metricName: "r2"})
future_mae = evaluator.evaluate(future_predictions, {evaluator.metricName: "mae"})

print(f"\nFuture Test (2024-2025) Results:")
print(f"  RMSE: {future_rmse:,.2f}")
print(f"  R2: {future_r2:.4f}")
print(f"  MAE: {future_mae:,.2f}")


Future Test (2024-2025) Results:
  RMSE: 95,682.99
  R2: 0.9396
  MAE: 16,210.43


## **12. Save Predictions**

In [35]:
predictions_sample = best_model.transform(test_data) \
    .select("label", "prediction", "property_type", "postcode_area", "year") \
    .limit(10000) \
    .toPandas()

predictions_sample.to_csv("../data/predictions_sample.csv", index=False)
print("Predictions sample saved")

Predictions sample saved


## **13. Export for Tableau**

In [36]:
tableau_export = results.copy()
tableau_export.to_csv("../tableau/model_performance.csv", index=False)

scalability_df.to_csv("../tableau/scalability_data.csv", index=False)

print("Tableau export files created")

Tableau export files created
